In [ ]:
#!pip install sentence-transformers

In [ ]:
#!pip install bertopic

In [ ]:
# =====================================================
# 1. 라이브러리
# =====================================================

import re
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

In [ ]:
# =====================================================
# 2. 데이터 불러오기
# =====================================================

INPUT_FILE = "liverpool_youtube_cafe_final_labeled.xlsx"
OUTPUT_FILE = "liverpool_cafe_bertopic_result.xlsx"

df = pd.read_excel(INPUT_FILE)

print("전체 데이터:", df.shape)
print(df.columns.tolist())

# 카페 데이터만 사용
cafe_df = df[df["source"] == "cafe"].copy()

print("카페 데이터:", cafe_df.shape)
print(cafe_df["period"].value_counts(dropna=False))
print(cafe_df["gpt_sentiment"].value_counts(dropna=False))

display(cafe_df.head())

In [ ]:
# =====================================================
# 3. 카페 데이터 기본 정리
# =====================================================

topic_df = cafe_df.copy()

# 감정 라벨 정리
topic_df["gpt_sentiment"] = (
    topic_df["gpt_sentiment"]
    .fillna("neutral")
    .astype(str)
    .str.lower()
    .str.strip()
    .replace({
        "긍정": "positive",
        "중립": "neutral",
        "부정": "negative"
    })
)

# period 정리
topic_df["topic_period"] = topic_df["period"].fillna("unknown").astype(str)

# 텍스트 정리
topic_df["text_clean"] = topic_df["text_clean"].fillna("").astype(str)

# 너무 짧은 텍스트 제거
topic_df = topic_df[topic_df["text_clean"].str.len() >= 5].reset_index(drop=True)

print("토픽모델링 대상 카페 데이터:", topic_df.shape)
print(topic_df["topic_period"].value_counts())
print(topic_df["gpt_sentiment"].value_counts())

In [ ]:
# =====================================================
# 4. 한국어 토큰화 설정
# =====================================================

try:
    from kiwipiepy import Kiwi
    kiwi = Kiwi()
    USE_KIWI = True
    print("✅ Kiwi 사용")
except:
    USE_KIWI = False
    print("⚠️ Kiwi 없음: 기본 정규식 토큰화 사용")

# =========================
# 리버풀 사용자 사전
# =========================

custom_words = [
    # 팀/팬덤
    "리버풀", "리버풀FC", "안필드", "콥", "더콥", "YNWA",

    # 감독/전술
    "슬롯", "아르네슬롯", "클롭", "위르겐클롭",

    # 주요 선수
    "알리송", "알리송베케르",
    "조고메즈", "고메즈",
    "엔도", "엔도와타루",
    "반다이크", "버질반다이크",
    "코나테",
    "케르케즈",
    "비르츠", "플로리안비르츠",
    "소보슬라이", "소보슬러이",
    "이삭", "이사크",
    "맥알리스터", "맥알",
    "살라", "모하메드살라",
    "브래들리",
    "키에사",
    "레오니",
    "커티스존스", "커존",
    "각포", "학포", "가크포",
    "에키티케",
    "마마르다슈빌리", "마마슈",
    "로버트슨",
    "우드먼",
    "프림퐁",
    "그라벤베르흐", "흐라벤베르흐",
    "트레이뇨니",
    "리오은구모하"

    # 조롱/밈 표현
    "리중딱", "살인풀", "전범풀", "뎀바바", "소라드",
    "훔바", "훔바훔바", "리버풀조롱"
]

for word in custom_words:
    try:
        kiwi.add_user_word(word, "NNP")
    except:
        pass

for word in custom_words:
    try:
        kiwi.add_user_word(word, "NNP")
    except:
        pass


stopwords = set([
    # [1] 기존 & 유튜브/커뮤니티 은어
    '진짜', '너무', '영상', '오늘', '경기', '그냥', '진심', 'ㅋㅋ', 'ㅎㅎ', 'ㅠㅠ', 'ㅜㅜ', '댓글', '개인',
    '이거', '저거', '그거', '뭔가', '정말', '지금', '다시', '계속', '가장', '제일', '조금', '정도','개추',
    
    # [2] 축구 일반 용어 (분석 인사이트가 없는 뻔한 단어)
    '축구', '하이라이트', '감독', '선수', '시즌', '리그', '전반', '후반', '골', '볼', '팬', '응원', '생각', '사람', '우리', '이번',
    
    # [3] 리버풀 지칭
    '리버풀', '리중딱', '전범풀',
    
    # [4] 타 팀 및 관련 선수 (노이즈 제거)
    '맨유', '맨체스터유나이티드', '맹구', '맨시티', '맨체스터시티', 
    '아스날', '아스널', '개스널', '리버풀', '전범풀', '첼시', '첼강딱', 
    '뉴캐슬', '빌라', '에버튼', '울버햄튼', '황희찬', 
    '레알', '바르샤', '뮌헨', '김민재', '이강인', '파리', 'psg', '리즈',
    '웨스트햄', '아스톤빌라','웨햄','클럽',
    
    
    # [6] 형태소 분석(Okt) 후 흔히 남는 무의미한 두 글자 명사들
    '때문', '위해', '대한', '관련', '진짜' ,"작성","짧은","글", "질문", "삭제", "필수", "검색", "경기시작", "익일", "오전" ,"일반","일반글", "작성", "금지","게시","필독"
])

stopwords = set(str(w).strip().lower() for w in stopwords)


def basic_clean_text(text):
    text = str(text)
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^가-힣a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def is_stopword(word):
    word = str(word).strip().lower()

    if word in stopwords:
        return True

    # 복합어 제거: 예) 리버풀팬, 게시글작성
    for sw in stopwords:
        if len(sw) >= 2 and sw in word:
            return True

    return False


def kiwi_tokenizer(text):
    text = basic_clean_text(text)
    tokens = []

    if USE_KIWI:
        for token in kiwi.tokenize(text):
            # NNG 일반명사, NNP 고유명사, SL 외국어
            if token.tag in ["NNG", "NNP", "SL"]:
                word = token.form.strip().lower()

                if len(word) >= 2 and not is_stopword(word):
                    tokens.append(word)

    else:
        raw_tokens = re.findall(r"[가-힣a-zA-Z]{2,}", text)
        tokens = [
            t.strip().lower()
            for t in raw_tokens
            if len(t.strip()) >= 2 and not is_stopword(t)
        ]

    return tokens

In [ ]:
# =====================================================
# 카페 게시글 상단 공지문 제거 함수
# =====================================================

NOTICE_PATTERNS = [

    # 게시글 하단 공지 문구
    r"짧은\s*글.*?삭제될\s*수\s*있습니다",
    r"질문\s*전\s*검색은\s*필수",
    r"상단의\s*필독.*?게시글\s*작성",

    # 규정 안내
    r"카페규칙\s*&\s*등업방법",
    r"의견\s*표현\s*기준\s*안내",
    r"정치\s*관련\s*게시글.*?운영\s*방침",
    r"선수별명\s*및\s*타팀별명관련",
    r"징계시\s*사유\s*문의",

    # 자주 나오는 안내문
    r"필독공지",
    r"카페\s*공지",
    r"공지사항"
]


def remove_cafe_notice(text):
    text = str(text)

    for pattern in NOTICE_PATTERNS:
        text = re.sub(pattern, " ", text)

    # 혹시 공지문이 한 줄로 붙어 있는 경우 추가 제거
    text = re.sub(
        r"짧은\s*글.*?게시글\s*작성",
        " ",
        text
    )

    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
# =====================================================
# 카페 공지문 제거 적용
# =====================================================

topic_df["topic_text"] = topic_df["text_clean"].fillna("").astype(str)

# cafe_post에만 공지문 제거
post_mask = topic_df["content_type"].eq("cafe_post")

topic_df.loc[post_mask, "topic_text"] = topic_df.loc[
    post_mask,
    "topic_text"
].apply(remove_cafe_notice)

# 너무 짧아진 텍스트 제거
topic_df = topic_df[
    topic_df["topic_text"].str.len() >= 5
].reset_index(drop=True)

print("공지문 제거 후 토픽모델링 대상:", topic_df.shape)

display(
    topic_df.loc[
        topic_df["content_type"].eq("cafe_post"),
        ["title", "text_clean", "topic_text"]
    ].head()
)

In [ ]:
# =========================
# 리버풀 표현 통합 사전
# =========================

custom_replace_dict = {
    # 팀명 통합
    "리버풀fc": "리버풀",
    "리버풀FC": "리버풀",
    "리버풀에프씨": "리버풀",
    "풀럼아님": "리버풀",

    # 팬덤/구장
    "더콥": "콥",
    "콥등이": "리버풀팬",
    "안필드": "안필드",
    "YNWA": "YNWA",
    "ynwa": "YNWA",

    # 감독명 통합
    "아르네슬롯": "슬롯",
    "슬롯감독": "슬롯",
    "클롭감독": "클롭",
    "위르겐클롭": "클롭",


    "알리송베케르": "알리송",
    "고메즈": "조고메즈",
    "엔도와타루": "엔도",
    "버질반다이크": "반다이크",
    "플로리안비르츠": "비르츠",
    "소보슬러이": "소보슬라이",
    "이사크": "이삭",
    "맥알": "맥알리스터",
    "모하메드살라": "살라",
    "커존": "커티스존스",
    "학포": "각포",
    "가크포": "각포",
    "마마슈": "마마르다슈빌리",
    "흐라벤베르흐": "그라벤베르흐",

    # 리버풀 조롱 표현 통합
    "리중딱": "리버풀조롱",
    "살인풀": "리버풀조롱",
    "전범풀": "리버풀조롱",
    "뎀바바": "리버풀조롱",
    "소라드": "리버풀조롱",
    "훔바": "리버풀조롱",
    "훔바훔바": "리버풀조롱",

    # 소보슬라이-제라드 조롱 관련
    "제라드후계자": "소보슬라이조롱",
    "제2의제라드": "소보슬라이조롱",
    "제라드DNA": "소보슬라이조롱",
    "제라드환생": "소보슬라이조롱",
    "미끄덩": "소보슬라이조롱",
    "미끄러짐": "소보슬라이조롱"
}

def normalize_football_terms(text):
    text = str(text)

    for src, tgt in custom_replace_dict.items():
        text = text.replace(src, tgt)

    return text

topic_df["topic_text"] = topic_df["topic_text"].apply(normalize_football_terms)

In [ ]:
# =====================================================
# 5. BERTopic 입력 문서 준비
# =====================================================

docs = topic_df["topic_text"].tolist()

print("문서 수:", len(docs))
print(docs[:3])

In [ ]:
# =====================================================
# 6. BERTopic 모델 설정
# =====================================================

# 한국어 문장 임베딩 모델
embedding_model = SentenceTransformer("jhgan/ko-sroberta-multitask")

# 문서 수에 따라 min_df 자동 조정
min_df_value = 2 if len(docs) < 3000 else 3

vectorizer_model = CountVectorizer(
    tokenizer=kiwi_tokenizer,
    token_pattern=None,
    min_df=min_df_value,
    max_df=0.85,
    max_features=7000
)

umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

hdbscan_model = HDBSCAN(
    min_cluster_size=20,
    min_samples=3,
    metric="euclidean",
    prediction_data=True
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    language="multilingual",
    calculate_probabilities=False,
    verbose=True
)

In [ ]:
# =====================================================
# 7. BERTopic 학습 실행
# =====================================================

topics, probs = topic_model.fit_transform(docs)

topic_df["topic"] = topics

print("토픽 부여 완료")
print(topic_df["topic"].value_counts().head(20))

In [ ]:
# =====================================================
# 8. Topic -1 줄이기
# =====================================================

print("기존 -1 개수:", (topic_df["topic"] == -1).sum())

try:
    new_topics = topic_model.reduce_outliers(
        docs,
        topics,
        strategy="c-tf-idf"
    )

    topic_model.update_topics(
        docs,
        topics=new_topics,
        vectorizer_model=vectorizer_model
    )

    topic_df["topic"] = new_topics

    print("수정 후 -1 개수:", (topic_df["topic"] == -1).sum())

except Exception as e:
    print("outlier reduction 실패:", e)

In [ ]:
# =====================================================
# 9. 토픽 정보 확인
# =====================================================

topic_info = topic_model.get_topic_info()

display(topic_info.head(30))

In [ ]:
# =====================================================
# 10. 토픽별 키워드 확인
# =====================================================

for topic_id in topic_info["Topic"].head(20):
    if topic_id == -1:
        continue

    print(f"\n===== Topic {topic_id} =====")
    print(topic_model.get_topic(topic_id))

In [ ]:
# =====================================================
# 11. 토픽별 대표 문장 확인
# =====================================================

for topic_id in topic_info["Topic"].head(10):
    if topic_id == -1:
        continue

    print(f"\n\n===== Topic {topic_id} 대표 문장 =====")

    rep_docs = topic_model.get_representative_docs(topic_id)

    for doc in rep_docs[:5]:
        print("-", doc[:200])

In [ ]:
# =====================================================
# 12. 토픽명 생성
# =====================================================

def make_topic_name(topic_id):
    if topic_id == -1:
        return "기타/분류불가"

    words = topic_model.get_topic(topic_id)

    if not words:
        return f"Topic {topic_id}"

    top_words = [w for w, score in words[:4]]

    return f"Topic {topic_id}: " + " / ".join(top_words)


topic_name_map = {
    topic_id: make_topic_name(topic_id)
    for topic_id in topic_df["topic"].unique()
}

topic_df["topic_name"] = topic_df["topic"].map(topic_name_map)

display(topic_df[["topic_period", "gpt_sentiment", "text_clean", "topic", "topic_name"]].head())

In [ ]:
# =====================================================
# 13. 38R 전후 토픽 비교
# =====================================================

valid_topic_df = topic_df[topic_df["topic"] != -1].copy()

period_topic_counts = pd.crosstab(
    valid_topic_df["topic_period"],
    valid_topic_df["topic_name"]
)

period_topic_ratio = pd.crosstab(
    valid_topic_df["topic_period"],
    valid_topic_df["topic_name"],
    normalize="index"
) * 100

display(period_topic_counts)
display(period_topic_ratio.round(2))

In [ ]:
# =====================================================
# 14. 38R 전후 Top 토픽 테이블
# =====================================================

def get_top_topics_by_group(data, group_col, top_n=10):
    result = []

    for group, group_df in data.groupby(group_col):
        counts = group_df["topic_name"].value_counts().head(top_n)
        total = len(group_df)

        for topic_name, count in counts.items():
            result.append({
                group_col: group,
                "topic_name": topic_name,
                "count": count,
                "ratio": round(count / total * 100, 2)
            })

    return pd.DataFrame(result)


period_top_topics = get_top_topics_by_group(
    valid_topic_df,
    "topic_period",
    top_n=10
)

display(period_top_topics)

In [ ]:
# =====================================================
# 15. 긍정 / 부정별 토픽 비교
# =====================================================

pn_topic_df = valid_topic_df[
    valid_topic_df["gpt_sentiment"].isin(["positive", "negative"])
].copy()

sentiment_topic_counts = pd.crosstab(
    pn_topic_df["gpt_sentiment"],
    pn_topic_df["topic_name"]
)

sentiment_topic_ratio = pd.crosstab(
    pn_topic_df["gpt_sentiment"],
    pn_topic_df["topic_name"],
    normalize="index"
) * 100

display(sentiment_topic_counts)
display(sentiment_topic_ratio.round(2))

In [ ]:
# =====================================================
# 16. 긍정/부정별 Top 토픽 테이블
# =====================================================

sentiment_top_topics = get_top_topics_by_group(
    pn_topic_df,
    "gpt_sentiment",
    top_n=10
)

display(sentiment_top_topics)

In [ ]:
# =====================================================
# 17. 한글 폰트 설정
# =====================================================

def set_korean_font():
    font_candidates = [
        "Malgun Gothic",
        "AppleGothic",
        "NanumGothic",
        "Noto Sans CJK KR"
    ]

    installed_fonts = [f.name for f in fm.fontManager.ttflist]

    for font in font_candidates:
        if font in installed_fonts:
            plt.rcParams["font.family"] = font
            plt.rcParams["axes.unicode_minus"] = False
            print(f"✅ 한글 폰트 설정 완료: {font}")
            return

    print("⚠️ 한글 폰트를 찾지 못했습니다.")

set_korean_font()

In [ ]:
# =====================================================
# 18. 카페 전체 Top 토픽 막대그래프
# =====================================================

top_topic_counts = (
    valid_topic_df["topic_name"]
    .value_counts()
    .head(12)
    .sort_values()
)

plt.figure(figsize=(10, 7))
plt.barh(top_topic_counts.index, top_topic_counts.values)
plt.title("팬 카페 주요 토픽 Top 12", fontsize=16, fontweight="bold")
plt.xlabel("문서 수")
plt.ylabel("토픽")
plt.grid(axis="x", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# =====================================================
# 19. 38R 전후 주요 토픽 비율 비교
# =====================================================

top_topics = (
    valid_topic_df["topic_name"]
    .value_counts()
    .head(8)
    .index
)

period_compare = pd.crosstab(
    valid_topic_df["topic_period"],
    valid_topic_df["topic_name"],
    normalize="index"
) * 100

period_compare = period_compare.reindex(index=["before_38R", "after_38R"])
period_compare = period_compare[top_topics].fillna(0)

display(period_compare.round(2))

period_compare.T.plot(
    kind="barh",
    figsize=(10, 7)
)

plt.title("팬 카페 38R 전후 주요 토픽 비율 비교", fontsize=16, fontweight="bold")
plt.xlabel("비율(%)")
plt.ylabel("토픽")
plt.grid(axis="x", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# =====================================================
# 20. 긍정/부정 주요 토픽 비율 비교
# =====================================================

top_topics_pn = (
    pn_topic_df["topic_name"]
    .value_counts()
    .head(8)
    .index
)

sentiment_compare = pd.crosstab(
    pn_topic_df["gpt_sentiment"],
    pn_topic_df["topic_name"],
    normalize="index"
) * 100

sentiment_compare = sentiment_compare.reindex(index=["positive", "negative"])
sentiment_compare = sentiment_compare[top_topics_pn].fillna(0)

display(sentiment_compare.round(2))

sentiment_compare.T.plot(
    kind="barh",
    figsize=(10, 7)
)

plt.title("팬 카페 긍정/부정 감정별 주요 토픽 비율 비교", fontsize=16, fontweight="bold")
plt.xlabel("비율(%)")
plt.ylabel("토픽")
plt.grid(axis="x", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# =====================================================
# 21. 결과 저장
# =====================================================

with pd.ExcelWriter(OUTPUT_FILE) as writer:
    topic_df.to_excel(writer, sheet_name="cafe_topic_labeled_data", index=False)
    topic_info.to_excel(writer, sheet_name="topic_info", index=False)
    period_top_topics.to_excel(writer, sheet_name="period_top_topics", index=False)
    sentiment_top_topics.to_excel(writer, sheet_name="sentiment_top_topics", index=False)
    period_topic_ratio.round(2).to_excel(writer, sheet_name="period_topic_ratio")
    sentiment_topic_ratio.round(2).to_excel(writer, sheet_name="sentiment_topic_ratio")

print("✅ 저장 완료:", OUTPUT_FILE)

In [ ]:
display(topic_info.head(20))
display(period_top_topics)
display(sentiment_top_topics)